# Part 4 · Generate the test split in every mode

Runs the trained model on the test split with the **4-bit transformer and the cached text embeddings** (no T5/CLIP on the GPU). Modes: sketch-only, reference-only, both without the Denoising-LoRA, both trained, plus a **text-only** FLUX baseline.

- **Needs:** Part 1 cache · both Condition-LoRAs · the Denoising-LoRA (Parts 2-3)
- **Produces:** `gen_r256.zip` in your store (images per mode)
- **Resumable:** images already written are skipped, so if the session dies, run the generation cell again.

In [ ]:
# --- bootstrap (identical in every part): find the pipeline files, set paths ---
import os, glob, shutil, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
    assert hits, 'Upload the pipeline/*.py files + pilot_v1.zip as a Kaggle dataset and add it as an input to this notebook.'
    SRC, WORK = os.path.dirname(hits[0]), '/kaggle/working'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC, WORK = '/content/drive/MyDrive/sketchy_dataset_one_sketch/pipeline', '/content'
os.makedirs(f'{WORK}/pipeline', exist_ok=True)
for f in glob.glob(f'{SRC}/*.py'):
    shutil.copy(f, f'{WORK}/pipeline/')
sys.path.insert(0, f'{WORK}/pipeline')
from common import *
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| STORE =', STORE)

In [ ]:
!pip install -q diffusers==0.33.1 transformers==4.52.4 peft==0.15.2 accelerate==1.7.0 bitsandbytes sentencepiece protobuf opencv-python-headless ipdb

In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
print(p.name, round(p.total_memory / 1e9), 'GB | bf16:', torch.cuda.get_device_capability()[0] >= 8)

In [ ]:
setup_repo()

In [ ]:
hf_login()   # FLUX.1-schnell is gated: accept the terms on its Hugging Face page first, then use a read token

In [ ]:
download('black-forest-labs/FLUX.1-schnell', FLUX, allow=DENOISE_PARTS)

In [ ]:
PILOT = f'{WORK}/pilot_v1'
if not os.path.exists(f'{PILOT}/metadata.jsonl'):
    shutil.unpack_archive(need('pilot_v1.zip'), PILOT)
print(open(f'{PILOT}/manifest.json').read()[:400])

In [ ]:
CACHE = need('cache_r256.pt')
CL = f'{CKPT}/Condition_LoRA'; DL = f'{CKPT}/Denoising_LoRA'
os.makedirs(CL, exist_ok=True); os.makedirs(DL, exist_ok=True)
for t in ['sketch', 'reference']:
    shutil.copy(need(f'Condition_LoRA/{t}.safetensors'), f'{CL}/{t}.safetensors')
shutil.rmtree(f'{DL}/sketch_reference_union', ignore_errors=True)
shutil.copytree(need('Denoising_LoRA/sketch_reference_union'), f'{DL}/sketch_reference_union')
print(os.listdir(CL), os.listdir(f'{DL}/sketch_reference_union'))

In [ ]:
import torch, time, unicombine_infer as UI
cache = torch.load(CACHE)
pipe = UI.load(UNI, FLUX, CL, DL, 'sketch_reference_union')
MODES = ['sketch', 'reference', 'both_free', 'both_trained']
GEN = f'{WORK}/gen'
LIMIT = 3          # first run: 3 test images per mode to check timing; then set to None for the whole split
for mode in MODES:
    t = time.time()
    UI.run_split(pipe, cache, PILOT, 'test', mode, f'{GEN}/{mode}', limit=LIMIT)
    print(mode, f'{(time.time() - t) / max(1, len(os.listdir(f"{GEN}/{mode}"))):.0f}s/image')

In [ ]:
# look at it: target | sketch | reference | sketch-only | reference-only | both (no Denoising-LoRA) | both (trained)
import json
from PIL import Image
from IPython.display import display
rows = [json.loads(l) for l in open(f'{PILOT}/metadata.jsonl')]
test = [r for r in rows if r['split'] == 'test'][:3]
S = 200
sheet = Image.new('RGB', (S * 7, S * len(test)), 'white')
for i, r in enumerate(test):
    ims = [Image.open(f"{PILOT}/{r[k]}") for k in ('target', 'sketch', 'reference')] + [Image.open(f"{GEN}/{m}/{r['id']}.png") for m in MODES]
    for j, im in enumerate(ims):
        sheet.paste(im.convert('RGB').resize((S, S)), (S * j, S * i))
display(sheet)

Looks sane? Set `LIMIT = None` in the generation cell and run it again for the full test split, then continue below. (Restart the generation cell as often as needed; finished images are skipped.)

In [ ]:
import gc
del pipe; gc.collect(); torch.cuda.empty_cache()
tp = UI.load_text_only(FLUX)
UI.run_split_text_only(tp, cache, PILOT, 'test', f'{GEN}/text_only', limit=LIMIT)
del tp; gc.collect(); torch.cuda.empty_cache()
print({m: len(os.listdir(f'{GEN}/{m}')) for m in ['text_only'] + MODES})

In [ ]:
shutil.make_archive(f'{WORK}/gen_r256', 'zip', GEN)
shutil.copy(f'{WORK}/gen_r256.zip', f'{STORE}/gen_r256.zip')
print('saved', f'{STORE}/gen_r256.zip')